In [0]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
import random

In [0]:
spark = SparkSession.builder.appName("DataAnalysis_PySpark").getOrCreate()

In [0]:
# Create schema

schema = StructType([
    StructField("altitude", ArrayType(DoubleType()), True),
    StructField("gender", StringType(), True),
    StructField("heart_rate", ArrayType(LongType()), True),
    StructField("id", LongType(), True),
    StructField("latitude", ArrayType(DoubleType()), True),
    StructField("longitude", ArrayType(DoubleType()), True),
    StructField("speed", ArrayType(DoubleType()), True),
    StructField("sport", StringType(), True),
    StructField("timestamp", ArrayType(LongType()), True),
    StructField("url", StringType(), True),
    StructField("userId", LongType(), True)
])

# Generate Data

def generate_data():
    sports = ["running", "cycling", "swimming", "hiking"]
    genders = ["male", "female", "other"]

    for x in range(201):
        yield(
           [random.uniform(0, 3000) for _ in range(5)],  # altitude
            random.choice(genders),                      # gender
            [random.randint(60, 180) for _ in range(5)], # heart_rate
            x,                                           # id
            [random.uniform(-90, 90) for _ in range(5)], # latitude
            [random.uniform(-180, 180) for _ in range(5)],# longitude
            [random.uniform(0, 50) for _ in range(5)],   # speed
            random.choice(sports),                      # sport
            [random.randint(1_600_000_000, 1_700_000_000) for _ in range(5)], # timestamp
            f"https://example.com/user_{x}",            # url
            random.randint(1, 1000)                     # userId 
        )

data = list(generate_data())
df = spark.createDataFrame(data, schema=schema)
df.show(5,truncate=True)



+--------------------+------+--------------------+---+--------------------+--------------------+--------------------+--------+--------------------+--------------------+------+
|            altitude|gender|          heart_rate| id|            latitude|           longitude|               speed|   sport|           timestamp|                 url|userId|
+--------------------+------+--------------------+---+--------------------+--------------------+--------------------+--------+--------------------+--------------------+------+
|[2650.61889980880...|  male|[166, 92, 102, 10...|  0|[-73.403682303789...|[-176.69346138309...|[35.3034754034894...|  hiking|[1616809150, 1694...|https://example.c...|   154|
|[2911.05646723212...|  male|[128, 130, 143, 1...|  1|[-46.363153632172...|[82.4525200828489...|[49.6702528983699...|swimming|[1647539611, 1623...|https://example.c...|   516|
|[1318.22351012856...|female|[118, 161, 160, 1...|  2|[77.1349383603781...|[-38.781213793406...|[44.3861865578854...| ru

In [0]:
# Create a temp view of df
df.createOrReplaceTempView('df_tbl')
spark.sql("SELECT * FROM df_tbl ").toPandas()


,altitude,gender,heart_rate,id,latitude,longitude,speed,sport,timestamp,url,userId
0,"[2650.6188998088087, 1142.662868095766, 954.22...",male,"[166, 92, 102, 100, 124]",0,"[-73.40368230378941, -46.27792997148453, -15.7...","[-176.69346138309632, -113.29188222673281, 53....","[35.303475403489486, 16.311238706681124, 49.84...",hiking,"[1616809150, 1694070279, 1633461143, 160473760...",https://example.com/user_0,154
1,"[2911.056467232124, 2383.7679691383, 1816.1748...",male,"[128, 130, 143, 165, 67]",1,"[-46.3631536321727, 76.80831866526228, -55.136...","[82.45252008284899, 176.29624598826655, 143.61...","[49.670252898369974, 13.350752799517624, 31.36...",swimming,"[1647539611, 1623947471, 1677116175, 162366024...",https://example.com/user_1,516
2,"[1318.2235101285669, 80.78850031359563, 2822.1...",female,"[118, 161, 160, 148, 146]",2,"[77.1349383603781, 45.89393363200054, -84.0505...","[-38.78121379340655, 77.74338607471651, 177.16...","[44.38618655788543, 31.674855704218434, 42.682...",running,"[1679982800, 1632796094, 1682667169, 168655971...",https://example.com/user_2,96
3,"[1444.2584551409086, 684.6802979154967, 2597.8...",male,"[158, 76, 160, 126, 151]",3,"[60.660773040831316, 46.20003371777446, -17.41...","[25.657233027470113, 31.847061904462123, -109....","[44.637101225394915, 38.794846663067815, 32.27...",running,"[1685610702, 1697905374, 1609044946, 163251072...",https://example.com/user_3,574
4,"[1470.6796419952486, 1362.9755335040945, 778.1...",other,"[126, 74, 112, 84, 167]",4,"[-0.5695303786698673, -60.353551656556476, 28....","[-0.41648823701379456, 109.53958137450928, 56....","[18.217000619361016, 26.185273120710345, 30.63...",running,"[1660828215, 1608793815, 1687072593, 162590169...",https://example.com/user_4,807
...,...,...,...,...,...,...,...,...,...,...,...
196,"[2100.992695426787, 1703.146264972777, 997.741...",other,"[124, 122, 86, 75, 175]",196,"[-36.397409905212406, -6.163019368038263, 79.9...","[57.2069641015525, -70.50214645341892, -29.528...","[37.17085622137308, 4.285965335158037, 15.9037...",swimming,"[1651335880, 1667275530, 1630283372, 169174066...",https://example.com/user_196,350
197,"[2582.07160718535, 2049.9991888096715, 1491.04...",female,"[114, 80, 88, 121, 70]",197,"[29.341045573701066, 14.982907656290635, -64.8...","[-1.1685357608990898, -7.6118880263127835, 30....","[41.32902042981622, 33.96560826232224, 13.6198...",hiking,"[1620385443, 1690911640, 1620278634, 160605569...",https://example.com/user_197,631
198,"[124.60197009473062, 1192.0469759602497, 861.6...",female,"[119, 177, 110, 165, 121]",198,"[-33.920092369435416, -81.91868727122414, -43....","[114.62662556753844, -70.83253195372228, 41.46...","[41.16681993266116, 9.435140111786072, 42.8764...",hiking,"[1647039714, 1695635972, 1676295351, 168567384...",https://example.com/user_198,943
199,"[492.90428288370236, 566.5156911568117, 236.16...",male,"[179, 141, 172, 61, 69]",199,"[58.29660409101183, 55.4896809988694, -7.42358...","[-73.27477623023015, 138.64930069659493, -100....","[8.595550527890827, 44.67669523808007, 43.8933...",swimming,"[1649818673, 1606084762, 1604920188, 165619021...",https://example.com/user_199,109


In [0]:
df.createOrReplaceTempView('df_tbl')
spark.sql("SELECT * FROM df_tbl ").show(5)


+--------------------+------+--------------------+---+--------------------+--------------------+--------------------+--------+--------------------+--------------------+------+
|            altitude|gender|          heart_rate| id|            latitude|           longitude|               speed|   sport|           timestamp|                 url|userId|
+--------------------+------+--------------------+---+--------------------+--------------------+--------------------+--------+--------------------+--------------------+------+
|[2650.61889980880...|  male|[166, 92, 102, 10...|  0|[-73.403682303789...|[-176.69346138309...|[35.3034754034894...|  hiking|[1616809150, 1694...|https://example.c...|   154|
|[2911.05646723212...|  male|[128, 130, 143, 1...|  1|[-46.363153632172...|[82.4525200828489...|[49.6702528983699...|swimming|[1647539611, 1623...|https://example.c...|   516|
|[1318.22351012856...|female|[118, 161, 160, 1...|  2|[77.1349383603781...|[-38.781213793406...|[44.3861865578854...| ru

In [0]:
# OverView Of DataFrame
print('Dataframe Overview')
df.printSchema()


Dataframe Overview
root
 |-- altitude: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- gender: string (nullable = true)
 |-- heart_rate: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- id: long (nullable = true)
 |-- latitude: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- longitude: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- speed: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- sport: string (nullable = true)
 |-- timestamp: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- url: string (nullable = true)
 |-- userId: long (nullable = true)



In [0]:
# Column Overview
pd.DataFrame(df.dtypes,columns =['column Name','Data Type'])

,column Name,Data Type
0,altitude,array<double>
1,gender,string
2,heart_rate,array<bigint>
3,id,bigint
4,latitude,array<double>
5,longitude,array<double>
6,speed,array<double>
7,sport,string
8,timestamp,array<bigint>
9,url,string


In [0]:
df.describe().show()

+-------+------+-----------------+--------+--------------------+-----------------+
|summary|gender|               id|   sport|                 url|           userId|
+-------+------+-----------------+--------+--------------------+-----------------+
|  count|   201|              201|     201|                 201|              201|
|   mean|  null|            100.0|    null|                null|501.6268656716418|
| stddev|  null|58.16786054171152|    null|                null|288.7362032628171|
|    min|female|                0| cycling|https://example.c...|                4|
|    max| other|              200|swimming|https://example.c...|              999|
+-------+------+-----------------+--------+--------------------+-----------------+



In [0]:
df.describe().toPandas()


,summary,gender,id,sport,url,userId
0,count,201,201,201,201,201
1,mean,None,100.0,None,None,501.6268656716418
2,stddev,None,58.16786054171152,None,None,288.7362032628171
3,min,female,0,cycling,https://example.com/user_0,4
4,max,other,200,swimming,https://example.com/user_99,999


In [0]:
df.toPandas()

,altitude,gender,heart_rate,id,latitude,longitude,speed,sport,timestamp,url,userId
0,"[2650.6188998088087, 1142.662868095766, 954.22...",male,"[166, 92, 102, 100, 124]",0,"[-73.40368230378941, -46.27792997148453, -15.7...","[-176.69346138309632, -113.29188222673281, 53....","[35.303475403489486, 16.311238706681124, 49.84...",hiking,"[1616809150, 1694070279, 1633461143, 160473760...",https://example.com/user_0,154
1,"[2911.056467232124, 2383.7679691383, 1816.1748...",male,"[128, 130, 143, 165, 67]",1,"[-46.3631536321727, 76.80831866526228, -55.136...","[82.45252008284899, 176.29624598826655, 143.61...","[49.670252898369974, 13.350752799517624, 31.36...",swimming,"[1647539611, 1623947471, 1677116175, 162366024...",https://example.com/user_1,516
2,"[1318.2235101285669, 80.78850031359563, 2822.1...",female,"[118, 161, 160, 148, 146]",2,"[77.1349383603781, 45.89393363200054, -84.0505...","[-38.78121379340655, 77.74338607471651, 177.16...","[44.38618655788543, 31.674855704218434, 42.682...",running,"[1679982800, 1632796094, 1682667169, 168655971...",https://example.com/user_2,96
3,"[1444.2584551409086, 684.6802979154967, 2597.8...",male,"[158, 76, 160, 126, 151]",3,"[60.660773040831316, 46.20003371777446, -17.41...","[25.657233027470113, 31.847061904462123, -109....","[44.637101225394915, 38.794846663067815, 32.27...",running,"[1685610702, 1697905374, 1609044946, 163251072...",https://example.com/user_3,574
4,"[1470.6796419952486, 1362.9755335040945, 778.1...",other,"[126, 74, 112, 84, 167]",4,"[-0.5695303786698673, -60.353551656556476, 28....","[-0.41648823701379456, 109.53958137450928, 56....","[18.217000619361016, 26.185273120710345, 30.63...",running,"[1660828215, 1608793815, 1687072593, 162590169...",https://example.com/user_4,807
...,...,...,...,...,...,...,...,...,...,...,...
196,"[2100.992695426787, 1703.146264972777, 997.741...",other,"[124, 122, 86, 75, 175]",196,"[-36.397409905212406, -6.163019368038263, 79.9...","[57.2069641015525, -70.50214645341892, -29.528...","[37.17085622137308, 4.285965335158037, 15.9037...",swimming,"[1651335880, 1667275530, 1630283372, 169174066...",https://example.com/user_196,350
197,"[2582.07160718535, 2049.9991888096715, 1491.04...",female,"[114, 80, 88, 121, 70]",197,"[29.341045573701066, 14.982907656290635, -64.8...","[-1.1685357608990898, -7.6118880263127835, 30....","[41.32902042981622, 33.96560826232224, 13.6198...",hiking,"[1620385443, 1690911640, 1620278634, 160605569...",https://example.com/user_197,631
198,"[124.60197009473062, 1192.0469759602497, 861.6...",female,"[119, 177, 110, 165, 121]",198,"[-33.920092369435416, -81.91868727122414, -43....","[114.62662556753844, -70.83253195372228, 41.46...","[41.16681993266116, 9.435140111786072, 42.8764...",hiking,"[1647039714, 1695635972, 1676295351, 168567384...",https://example.com/user_198,943
199,"[492.90428288370236, 566.5156911568117, 236.16...",male,"[179, 141, 172, 61, 69]",199,"[58.29660409101183, 55.4896809988694, -7.42358...","[-73.27477623023015, 138.64930069659493, -100....","[8.595550527890827, 44.67669523808007, 43.8933...",swimming,"[1649818673, 1606084762, 1604920188, 165619021...",https://example.com/user_199,109


In [0]:
df.count()

Out[129]: 201

In [0]:
df.toPandas().head(5)

,altitude,gender,heart_rate,id,latitude,longitude,speed,sport,timestamp,url,userId
0,"[2650.6188998088087, 1142.662868095766, 954.22...",male,"[166, 92, 102, 100, 124]",0,"[-73.40368230378941, -46.27792997148453, -15.7...","[-176.69346138309632, -113.29188222673281, 53....","[35.303475403489486, 16.311238706681124, 49.84...",hiking,"[1616809150, 1694070279, 1633461143, 160473760...",https://example.com/user_0,154
1,"[2911.056467232124, 2383.7679691383, 1816.1748...",male,"[128, 130, 143, 165, 67]",1,"[-46.3631536321727, 76.80831866526228, -55.136...","[82.45252008284899, 176.29624598826655, 143.61...","[49.670252898369974, 13.350752799517624, 31.36...",swimming,"[1647539611, 1623947471, 1677116175, 162366024...",https://example.com/user_1,516
2,"[1318.2235101285669, 80.78850031359563, 2822.1...",female,"[118, 161, 160, 148, 146]",2,"[77.1349383603781, 45.89393363200054, -84.0505...","[-38.78121379340655, 77.74338607471651, 177.16...","[44.38618655788543, 31.674855704218434, 42.682...",running,"[1679982800, 1632796094, 1682667169, 168655971...",https://example.com/user_2,96
3,"[1444.2584551409086, 684.6802979154967, 2597.8...",male,"[158, 76, 160, 126, 151]",3,"[60.660773040831316, 46.20003371777446, -17.41...","[25.657233027470113, 31.847061904462123, -109....","[44.637101225394915, 38.794846663067815, 32.27...",running,"[1685610702, 1697905374, 1609044946, 163251072...",https://example.com/user_3,574
4,"[1470.6796419952486, 1362.9755335040945, 778.1...",other,"[126, 74, 112, 84, 167]",4,"[-0.5695303786698673, -60.353551656556476, 28....","[-0.41648823701379456, 109.53958137450928, 56....","[18.217000619361016, 26.185273120710345, 30.63...",running,"[1660828215, 1608793815, 1687072593, 162590169...",https://example.com/user_4,807


In [0]:
df.toPandas().tail(5)

,altitude,gender,heart_rate,id,latitude,longitude,speed,sport,timestamp,url,userId
196,"[2100.992695426787, 1703.146264972777, 997.741...",other,"[124, 122, 86, 75, 175]",196,"[-36.397409905212406, -6.163019368038263, 79.9...","[57.2069641015525, -70.50214645341892, -29.528...","[37.17085622137308, 4.285965335158037, 15.9037...",swimming,"[1651335880, 1667275530, 1630283372, 169174066...",https://example.com/user_196,350
197,"[2582.07160718535, 2049.9991888096715, 1491.04...",female,"[114, 80, 88, 121, 70]",197,"[29.341045573701066, 14.982907656290635, -64.8...","[-1.1685357608990898, -7.6118880263127835, 30....","[41.32902042981622, 33.96560826232224, 13.6198...",hiking,"[1620385443, 1690911640, 1620278634, 160605569...",https://example.com/user_197,631
198,"[124.60197009473062, 1192.0469759602497, 861.6...",female,"[119, 177, 110, 165, 121]",198,"[-33.920092369435416, -81.91868727122414, -43....","[114.62662556753844, -70.83253195372228, 41.46...","[41.16681993266116, 9.435140111786072, 42.8764...",hiking,"[1647039714, 1695635972, 1676295351, 168567384...",https://example.com/user_198,943
199,"[492.90428288370236, 566.5156911568117, 236.16...",male,"[179, 141, 172, 61, 69]",199,"[58.29660409101183, 55.4896809988694, -7.42358...","[-73.27477623023015, 138.64930069659493, -100....","[8.595550527890827, 44.67669523808007, 43.8933...",swimming,"[1649818673, 1606084762, 1604920188, 165619021...",https://example.com/user_199,109
200,"[1160.436654029477, 1687.5654848062832, 492.44...",male,"[64, 83, 153, 109, 147]",200,"[76.83027996668844, 0.5402026124079953, -83.55...","[-155.3308626648845, 66.11643432086024, 74.252...","[13.377989473855568, 0.6328065231673119, 41.19...",cycling,"[1687905187, 1693451270, 1653823781, 164580006...",https://example.com/user_200,428


In [0]:
df.columns

Out[132]: ['altitude',
 'gender',
 'heart_rate',
 'id',
 'latitude',
 'longitude',
 'speed',
 'sport',
 'timestamp',
 'url',
 'userId']

In [0]:
df.dtypes

Out[133]: [('altitude', 'array<double>'),
 ('gender', 'string'),
 ('heart_rate', 'array<bigint>'),
 ('id', 'bigint'),
 ('latitude', 'array<double>'),
 ('longitude', 'array<double>'),
 ('speed', 'array<double>'),
 ('sport', 'string'),
 ('timestamp', 'array<bigint>'),
 ('url', 'string'),
 ('userId', 'bigint')]

In [0]:
'''
CHECK FOR :
    1.For string columns, we check for None and null
    2.For numeric columns, we check for zeroes and NaN
    3.For array type columns, we check if the array contain zeroes or NaN
'''
# Create a list of String Column
stringColumn = ['gender','sport','url']
# Create a list of Numeric Column
numericColumn = ['id','userID']
## Create a list of array Column
arrayColumn = ['altitude', 'heart_rate', 'latitude', 'longitude', 'speed', 'timestamp']

# create a dict to store missing values = {}
missing_val = {}

# run loop to check missing val 
for index,column in enumerate(df.columns):
    if column in stringColumn:
        missing_count_str = df.filter(col(column).eqNullSafe(None) \
            | col(column).isNull()
            ).count()
        missing_val.update({column:missing_count_str})
    if column in numericColumn:
        missing_count_num = df.where(col(column).isin([0,None,np.nan])).count()
        missing_val.update({column:missing_count_num})
    if column in arrayColumn:
        missing_count_arr = df.filter(array_contains(df[column],0)\
            | array_contains(df[column],np.nan)).count()
        missing_val.update({column:missing_count_arr})
missing_df = pd.DataFrame.from_dict([missing_val])
missing_df



,altitude,gender,heart_rate,id,latitude,longitude,speed,sport,timestamp,url
0,0,0,0,1,0,0,0,0,0,0


In [0]:
df.groupBy("sport").count().show()

+--------+-----+
|   sport|count|
+--------+-----+
|  hiking|   48|
| cycling|   57|
| running|   54|
|swimming|   42|
+--------+-----+



In [0]:
df.printSchema()

root
 |-- altitude: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- gender: string (nullable = true)
 |-- heart_rate: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- id: long (nullable = true)
 |-- latitude: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- longitude: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- speed: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- sport: string (nullable = true)
 |-- timestamp: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- url: string (nullable = true)
 |-- userId: long (nullable = true)



In [0]:
df.groupBy('sport').agg(
    count('id').alias('Count'),\
    countDistinct('id').alias('Distinct_Count')
    ).show()

+--------+-----+--------------+
|   sport|Count|Distinct_Count|
+--------+-----+--------------+
|  hiking|   48|            48|
| cycling|   57|            57|
| running|   54|            54|
|swimming|   42|            42|
+--------+-----+--------------+



In [0]:
df.select('timestamp').show()

+--------------------+
|           timestamp|
+--------------------+
|[1616809150, 1694...|
|[1647539611, 1623...|
|[1679982800, 1632...|
|[1685610702, 1697...|
|[1660828215, 1608...|
|[1655667979, 1644...|
|[1697112749, 1618...|
|[1690235164, 1695...|
|[1642489591, 1650...|
|[1631831035, 1635...|
|[1657666037, 1669...|
|[1686957003, 1658...|
|[1642050418, 1652...|
|[1670238829, 1657...|
|[1629044243, 1614...|
|[1682457532, 1678...|
|[1697210049, 1689...|
|[1693996443, 1605...|
|[1635837003, 1665...|
|[1654611032, 1617...|
+--------------------+
only showing top 20 rows



In [0]:
# We create new column to count the number of timestamps recorded per row/workout, named as 'PerWorkoutRecordCount' column
df=df.withColumn('PerWorkoutRecordCount',size(col('timestamp')))
df.printSchema()

root
 |-- altitude: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- gender: string (nullable = true)
 |-- heart_rate: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- id: long (nullable = true)
 |-- latitude: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- longitude: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- speed: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- sport: string (nullable = true)
 |-- timestamp: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- url: string (nullable = true)
 |-- userId: long (nullable = true)
 |-- PerWorkoutRecordCount: integer (nullable = false)



In [0]:
df.select('PerWorkoutRecordCount').show()

+---------------------+
|PerWorkoutRecordCount|
+---------------------+
|                    5|
|                    5|
|                    5|
|                    5|
|                    5|
|                    5|
|                    5|
|                    5|
|                    5|
|                    5|
|                    5|
|                    5|
|                    5|
|                    5|
|                    5|
|                    5|
|                    5|
|                    5|
|                    5|
|                    5|
+---------------------+
only showing top 20 rows



In [0]:
def user_activity_workout_summary (df):
    user_count = df.select('userID').distinct().count()
    workout_count = df.select('id').distinct().count()
    activity_count = df.select('sport').distinct().count()
    sum_temp = df.agg(sum('PerWorkoutRecordCount'))
    total_count = df.count()
    columns=['Users count','Workouts count','Activity types count','PerWorkoutRecordCount', 'Total records count']
    data = [[user_count],[workout_count],[activity_count],[sum_temp],[total_count]]
    summary_dict = { column : data[i] for i ,column in enumerate(columns)}
    summary_df = pd.DataFrame.from_dict(summary_dict)[columns]
    return summary_df

summaryDF = user_activity_workout_summary (df)
summaryDF


,Users count,Workouts count,Activity types count,PerWorkoutRecordCount,Total records count
0,184,201,4,DataFrame[sum(PerWorkoutRecordCount): bigint],201


In [0]:
gender_user_count = df.select('gender','userId').distinct().groupBy('gender').count().toPandas()
gender_activities_count = df.groupBy('gender').count().toPandas()

In [0]:

# Joining 2 df
gender_user_activity_count = gender_user_count.join(
        gender_activities_count.set_index('gender'), on='gender'
        , how='inner', lsuffix='_gu'
    )

gender_user_activity_count

,gender,count_gu,count
0,female,62,64
1,other,67,67
2,male,66,70


In [0]:
df.groupBy('gender').agg(
    countDistinct('id').alias('distinct'),
    count('id').alias('count')
).show()

+------+--------+-----+
|gender|distinct|count|
+------+--------+-----+
|female|      64|   64|
| other|      67|   67|
|  male|      70|   70|
+------+--------+-----+



In [0]:
df.printSchema()

root
 |-- altitude: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- gender: string (nullable = true)
 |-- heart_rate: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- id: long (nullable = true)
 |-- latitude: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- longitude: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- speed: array (nullable = true)
 |    |-- element: double (containsNull = true)
 |-- sport: string (nullable = true)
 |-- timestamp: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- url: string (nullable = true)
 |-- userId: long (nullable = true)
 |-- PerWorkoutRecordCount: integer (nullable = false)



In [0]:
df.select('sport').distinct().show()

+--------+
|   sport|
+--------+
|  hiking|
| cycling|
| running|
|swimming|
+--------+



In [0]:
# Filtering 
df_filter = df.select('id','sport','url').where(df.sport == 'swimming')
df_filter.show(5)

+---+--------+--------------------+
| id|   sport|                 url|
+---+--------+--------------------+
|  1|swimming|https://example.c...|
| 11|swimming|https://example.c...|
| 35|swimming|https://example.c...|
| 50|swimming|https://example.c...|
| 51|swimming|https://example.c...|
+---+--------+--------------------+
only showing top 5 rows



In [0]:
df_filter = df_filter.withColumnRenamed('sport','play_sport')
df_filter.show(5)

+---+----------+--------------------+
| id|play_sport|                 url|
+---+----------+--------------------+
|  1|  swimming|https://example.c...|
| 11|  swimming|https://example.c...|
| 35|  swimming|https://example.c...|
| 50|  swimming|https://example.c...|
| 51|  swimming|https://example.c...|
+---+----------+--------------------+
only showing top 5 rows

